In [10]:
%pip install pandas numpy scikit-learn lightgbm

  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-4.7.0-py3-none-macosx_12_0_arm64.whl.metadata (18 kB)
  Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl (8.2 MB)
Using cached lightgbm-4.7.0-py3-none-macosx_12_0_arm64.whl (1.5 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl (20.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [lightgbm]4/6 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
print("데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)")
train = pd.read_csv('./data/train.csv')

# 2. 결측치 처리 (이전에 계획한 단순 대체 방법)
print("결측치 처리 중...")
# 숫자형 결측치는 평균으로, 범주형 결측치는 'Unknown'으로 채웁니다.
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna(train[col].mean())

object_cols = train.select_dtypes(include=['object']).columns
for col in object_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna('Unknown')

# 3. Train / Validation 데이터 나누기 (매우 중요!)
print("학습용/검증용 데이터 분리 중...")
# 대회 평가에 제출할 'row_id'는 예측에 방해되므로 제외하고, 정답지인 'control_success'도 분리합니다.
X = train.drop(columns=['control_success', 'row_id'])
y = train['control_success']

# 모델 학습용 80%, 로컬 검증용 20%로 나눕니다.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42 # 항상 동일하게 나뉘도록 시드 고정
)

print(f"✅ 완료! 학습용 데이터 크기: {X_train.shape}, 검증용 데이터 크기: {X_valid.shape}")


데이터 로딩 중... (용량이 커서 몇 초 정도 걸릴 수 있습니다)
결측치 처리 중...
학습용/검증용 데이터 분리 중...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_21116/3416133666.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train.select_dtypes(include=['object']).columns


✅ 완료! 학습용 데이터 크기: (1180073, 47), 검증용 데이터 크기: (295019, 47)


In [13]:
import lightgbm as lgb
from sklearn.metrics import brier_score_loss

print("1. 범주형 데이터 변환 중 (글자를 모델이 이해할 수 있게 변경)...")
# 아까 'Unknown'으로 채웠던 글자(Object) 데이터들을 LightGBM이 좋아하는 'category' 타입으로 변경합니다.
object_cols = X_train.select_dtypes(include=['object']).columns
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

print("2. LightGBM 모델 세팅 및 학습 시작 (시간이 약간 소요됩니다)...")
# 예측 속도가 빠르고 성능이 좋은 LightGBM 모델을 준비합니다.
model = lgb.LGBMClassifier(
    n_estimators=100,      # 나무(Tree)를 100개 만듭니다.
    random_state=42,       # 매번 동일한 결과가 나오도록 고정
    n_jobs=-1              # 컴퓨터의 모든 코어를 사용해서 빠르게 학습
)

# 모델 학습 (공부 시작!)
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

print("\n3. 모의고사(검증 데이터) 예측 및 채점 중...")
# 🚨핵심: 제구 성공 '확률'을 구해야 하므로 predict가 아닌 predict_proba를 사용합니다!
# [:, 1]은 제구 실패(0), 성공(1) 중 성공할(1) 확률값만 가져온다는 뜻입니다.
y_pred_proba = model.predict_proba(X_valid)[:, 1]

# 4. 대회 공식 산식(Brier Skill Score)으로 자체 점수 계산
brier = brier_score_loss(y_valid, y_pred_proba)
r = y_train.mean() # 학습 데이터의 전체 평균 제구 성공률
brier_mean = r * (1 - r)
bss = max(0, 100000 * (1 - brier / brier_mean))

print("="*40)
print("🎯 학습 및 자체 평가 완료!")
print(f"▶ 순수 Brier Score (낮을수록 좋음): {brier:.4f}")
print(f"🏆 예상 환산 점수 (높을수록 좋음, 최대 10만점): {bss:.2f}점")
print("="*40)

1. 범주형 데이터 변환 중 (글자를 모델이 이해할 수 있게 변경)...
2. LightGBM 모델 세팅 및 학습 시작 (시간이 약간 소요됩니다)...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_21116/92834513.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(include=['object']).columns
/opt/homebrew/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 618302, number of negative: 561771
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025692 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6389
[LightGBM] [Info] Number of data points in the train set: 1180073, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523952 -> initscore=0.095883
[LightGBM] [Info] Start training from score 0.095883
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.682845

3. 모의고사(검증 데이터) 예측 및 채점 중...
🎯 학습 및 자체 평가 완료!
▶ 순수 Brier Score (낮을수록 좋음): 0.2449
🏆 예상 환산 점수 (높을수록 좋음, 최대 10만점): 1797.60점


In [18]:
import joblib
import os

# model 폴더가 없다면 생성합니다.
os.makedirs('./model', exist_ok=True)

# 아까 학습시킨 LightGBM 모델을 'model.pkl'이라는 이름으로 저장합니다.
joblib.dump(model, './model/model.pkl')
print("✅ 모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다!")

✅ 모델이 ./model/model.pkl 경로에 성공적으로 저장되었습니다!


In [19]:
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("lightgbm\n")
print("✅ requirements.txt 파일이 성공적으로 생성되었습니다!")

✅ requirements.txt 파일이 성공적으로 생성되었습니다!


In [20]:
script_content = """import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os

def main():
    print("1. 평가 데이터 불러오기...")
    # 🚨 평가 서버 환경에 맞춰 상대 경로 지정
    test = pd.read_csv('./data/test.csv')
    
    # 제출 양식에 넣을 row_id를 따로 빼두고, 모델에 넣을 피처(X_test)만 남깁니다.
    row_ids = test['row_id']
    X_test = test.drop(columns=['row_id'])
    
    print("2. 데이터 전처리 진행 중...")
    # 숫자형(Numeric) 결측치 처리: 에러 방지를 위해 단순 0으로 채웁니다.
    # (원래는 Train 데이터의 평균을 쓰는 것이 정석입니다.)
    numeric_cols = X_test.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna(0)
            
    # 범주형(Object) 결측치 처리 및 LightGBM용 category 타입 변환
    object_cols = X_test.select_dtypes(include=['object']).columns
    for col in object_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna('Unknown')
        X_test[col] = X_test[col].astype('category')
        
    print("3. 나만의 LightGBM 모델 불러오기...")
    # 우리가 훈련시켜서 ./model/ 폴더에 저장해둔 모델을 깨웁니다.
    model = joblib.load('./model/model.pkl')
    
    print("4. 제구 성공 확률 예측 중...")
    # predict_proba를 사용하여 0~1 사이의 확률값을 뽑아냅니다.
    # [:, 1]은 '제구 실패(0)', '성공(1)' 중 '성공(1)'에 대한 확률만 가져온다는 뜻입니다.
    pred_proba = model.predict_proba(X_test)[:, 1]
    
    print("5. 예측 결과 저장 중...")
    # 서버 요구사항에 맞게 output 폴더를 만들고 그 안에 submission.csv를 저장합니다.
    os.makedirs('./output', exist_ok=True)
    submission = pd.DataFrame({
        'row_id': row_ids,
        'control_success': pred_proba
    })
    submission.to_csv('./output/submission.csv', index=False)
    print("✅ 모든 추론 과정이 완료되었습니다!")

if __name__ == '__main__':
    main()
"""

# 위 내용을 script.py 파일로 저장 (이미 있다면 덮어쓰기 됨)
with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
    
print("✅ 제출용 script.py 파일이 완벽하게 새로 작성되었습니다!")

✅ 제출용 script.py 파일이 완벽하게 새로 작성되었습니다!
